In [2]:
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup, Comment
import re
import os
from pathlib import Path

In [3]:
games_path = Path("/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Games")

### Needed IDs
* Expected Points Summary = expected_points
* Team Stats = team_stats
* Passing, Rushing, and Receiving = player_offense
* Defense = player_defense
* Kick/Punt Return = returns
* Kicking & Punting = kicking
* Advanced Passing = passing_advanced
* Advanced Receiving = receiving_advanced
* Advanced Rushing = rushing_advanced
* Advanced Defense = defense_advanced
* Home Team Drives = home_drives
* Away Team Drives = vis_drives

#### Expected Points Summary

In [3]:
expected_pts_df = []
for game in os.listdir(games_path):
    with open(f"/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Games/{game}", encoding="utf-8") as f:
        page = f.read()

    soup = BeautifulSoup(page, "html.parser")

    scorebox_meta = soup.find("div", class_="scorebox_meta")
    game_date = scorebox_meta.find_all("div")[0].text.strip()

    comment = next(c for c in soup.find_all(string=lambda text: isinstance(text, Comment)) if "expected_points" in c)

    table = BeautifulSoup(comment, "html.parser").find(id="expected_points")
    df = pd.read_html(StringIO(str(table)))[0]
    df["game_date"]=game_date
    df["game_id"]=game
    expected_pts_df.append(df)

expected_pts_df_combined = pd.concat(expected_pts_df, ignore_index=True)

In [ ]:
expected_pts_df_combined

#### Team Stats

In [7]:
team_stats_df = []
for game in os.listdir(games_path):
    with open(f"/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Games/{game}", encoding="utf-8") as f:
        page = f.read()
    
    soup = BeautifulSoup(page, "html.parser")

    scorebox_meta = soup.find("div", class_="scorebox_meta")
    game_date = scorebox_meta.find_all("div")[0].text.strip()

    comment = next(c for c in soup.find_all(
        string=lambda text: isinstance(text, Comment)) if "team_stats" in c)

    table = BeautifulSoup(comment, "html.parser").find(id="team_stats")
    df = pd.read_html(StringIO(str(table)))[0]
    df["date"]= game_date

    #Fix Parse Structure
    team_one = df.columns[1]
    team_two = df.columns[2]
    date_value = df.iat[0,1]

    pivot_one = df.pivot(columns="Unnamed: 0", values=team_one)
    pivot_one["team_name"] = team_one
    pivot_one = pivot_one.bfill().iloc[[0]]
    
    pivot_two = df.pivot(columns="Unnamed: 0", values=team_two)
    pivot_two["team_name"] = team_two
    pivot_two = pivot_two.bfill().iloc[[0]]

    team_stats_df.append(pivot_one)
    team_stats_df.append(pivot_two)

team_stats_df = pd.concat(team_stats_df, ignore_index=True)

In [8]:
team_stats_df

Unnamed: 0,Cmp-Att-Yd-TD-INT,First Downs,Fourth Down Conv.,Fumbles-Lost,Net Pass Yards,Penalties-Yards,Rush-Yds-TDs,Sacked-Yards,Third Down Conv.,Time of Possession,Total Yards,Turnovers,team_name
0,23-43-291-1-3,19,1-4,2-0,283,8-81,25-106-0,1-8,8-15,34:42,389,3,GNB
1,14-26-137-2-1,19,0-2,0-0,137,7-62,31-117-0,0-0,6-11,25:18,254,1,DET
2,15-28-154-1-1,20,3-3,2-1,127,10-91,39-211-2,4-27,3-16,36:18,338,2,PHI
3,24-39-258-5-2,23,2-3,3-3,255,6-93,24-113-0,1-3,7-13,23:42,368,5,WAS
4,27-41-204-2-0,23,0-0,0-0,204,5-45,27-107-1,0-0,7-16,31:21,311,0,DAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,17-31-148-1-0,16,0-2,2-1,119,3-20,23-74-1,4-29,4-12,22:00,193,1,PIT
2274,20-32-253-2-0,19,2-2,1-1,242,6-38,25-97-0,4-11,4-12,31:02,339,1,DAL
2275,24-37-293-1-0,22,0-1,1-0,284,5-33,26-91-0,1-9,6-13,28:58,375,0,MIA
2276,22-32-191-1-1,18,1-2,1-0,171,4-25,28-93-1,3-20,3-12,26:56,264,1,WAS


In [12]:
player_offense_df = []
for game in os.listdir(games_path):
    with open(f"/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Games/{game}", encoding="utf-8") as f:
        page = f.read()

    soup = BeautifulSoup(page, "html.parser")

    scorebox_meta = soup.find("div", class_="scorebox_meta")
    game_date = scorebox_meta.find_all("div")[0].text.strip()

    tables = soup.find_all(id="player_offense")

    for table in tables:
        table_html = str(table)
        df = pd.read_html(StringIO(table_html))[0]
        df["date"] = game_date
        player_offense_df.append(df)

player_offense_df = pd.concat(player_offense_df, ignore_index=True)

In [13]:
player_offense_df

Unnamed: 0_level_0 Unnamed: 1_level_0 Passing                           \
                  Player                 Tm     Cmp Att  Yds TD Int Sk Yds.1   
0          Aaron Rodgers                GNB      23  43  291  1   3  1     8   
1              AJ Dillon                GNB       0   0    0  0   0  0     0   
2            Aaron Jones                GNB       0   0    0  0   0  0     0   
3             Kylin Hill                GNB       0   0    0  0   0  0     0   
4           Allen Lazard                GNB       0   0    0  0   0  0     0   
...                  ...                ...     ...  ..  ... ..  .. ..   ...   
24798   Kenneth Gainwell                PHI       0   0    0  0   0  0     0   
24799         A.J. Brown                PHI       0   0    0  0   0  0     0   
24800     Dallas Goedert                PHI       0   0    0  0   0  0     0   
24801      DeVonta Smith                PHI       0   0    0  0   0  0     0   
24802       Jahan Dotson                PHI       0   0    0  0   0  0     0   

           ... Rushing     Receiving                Fumbles     \
      Lng  ...      TD Lng       Tgt Rec Yds TD Lng     Fmb FL   
0      47  ...       0  18         0   0   0  0   0       0  0   
1       0  ...       0   9         4   2  10  0   7       1  0   
2       0  ...       0   9         2   2  20  0  15       0  0   
3       0  ...       0   7         0   0   0  0   0       0  0   
4       0  ...       0   0        10   4  87  1  47       0  0   
...    ..  ...     ...  ..       ...  ..  .. ..  ..     ... ..   
24798   0  ...       0  14         1   1   6  0   6       0  0   
24799   0  ...       0   0         8   5  65  0  25       0  0   
24800   0  ...       0   0         5   5  61  0  32       1  0   
24801   0  ...       0   0         6   4  29  0  21       0  0   
24802   0  ...       0   0         2   1   8  0   8       0  0   

                        date  
                              
0         Sunday Nov 6, 2022  
1         Sunday Nov 6, 2022  
2         Sunday Nov 6, 2022  
3         Sunday Nov 6, 2022  
4         Sunday Nov 6, 2022  
...                      ...  
24798  Thursday Nov 14, 2024  
24799  Thursday Nov 14, 2024  
24800  Thursday Nov 14, 2024  
24801  Thursday Nov 14, 2024  
24802  Thursday Nov 14, 2024  

[24803 rows x 23 columns]

In [14]:
player_defense_df = []
for game in os.listdir(games_path):
    with open(f"/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Games/{game}", encoding="utf-8") as f:
        page = f.read()
    
    soup = BeautifulSoup(page, "html.parser")

    scorebox_meta = soup.find("div", class_="scorebox_meta")
    game_date = scorebox_meta.find_all("div")[0].text.strip()

    comment = next(c for c in soup.find_all(
        string=lambda text: isinstance(text, Comment)) if "player_defense" in c)

    table = BeautifulSoup(comment, "html.parser").find(id="player_defense")
    df = pd.read_html(StringIO(str(table)))[0]
    df["date"]= game_date
    player_defense_df.append(df)

player_defense_df = pd.concat(player_defense_df, ignore_index=True)

In [15]:
player_defense_df

Unnamed: 0_level_0 Unnamed: 1_level_0 Def Interceptions             \
                     Player                 Tm               Int Yds TD Lng   
0           Jaire Alexander                GNB                 1  29  0  29   
1               Krys Barnes                GNB                 0   0  0   0   
2               Kenny Clark                GNB                 0   0  0   0   
3               Adrian Amos                GNB                 0   0  0   0   
4             Rasul Douglas                GNB                 0   0  0   0   
...                     ...                ...               ...  .. ..  ..   
47540      Quinyon Mitchell                PHI                 0   0  0   0   
47541            Moro Ojomo                PHI                 0   0  0   0   
47542  Jeremiah Trotter Jr.                PHI                 0   0  0   0   
47543       Milton Williams                PHI                 0   0  0   0   
47544      Grant Calcaterra                PHI                 0   0  0   0   

         Unnamed: 7_level_0 Tackles                     Fumbles            \
      PD                 Sk    Comb Solo Ast TFL QBHits      FR Yds TD FF   
0      1                0.0       2    2   0   0      0       0   0  0  0   
1      0                0.0       8    4   4   0      0       0   0  0  0   
2      0                0.0       7    4   3   0      0       0   0  0  0   
3      0                0.0       5    3   2   0      0       0   0  0  0   
4      0                0.0       5    4   1   0      0       0   0  0  0   
...   ..                ...     ...  ...  ..  ..    ...     ...  .. .. ..   
47540  0                0.0       1    0   1   0      0       0   0  0  0   
47541  0                0.0       1    1   0   0      0       0   0  0  0   
47542  0                0.0       1    1   0   0      0       0   0  0  0   
47543  0                0.0       1    0   1   0      0       0   0  0  0   
47544  0                0.0       0    0   0   0      0       1   2  0  0   

                        date  
                              
0         Sunday Nov 6, 2022  
1         Sunday Nov 6, 2022  
2         Sunday Nov 6, 2022  
3         Sunday Nov 6, 2022  
4         Sunday Nov 6, 2022  
...                      ...  
47540  Thursday Nov 14, 2024  
47541  Thursday Nov 14, 2024  
47542  Thursday Nov 14, 2024  
47543  Thursday Nov 14, 2024  
47544  Thursday Nov 14, 2024  

[47545 rows x 18 columns]